In [1]:
import pandas as pd
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn  # or mlflow.xgboost
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, roc_auc_score
import time
import os
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from feature_engine.outliers import OutlierTrimmer

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_csv(r"C:\projects\customer churn prediction\Data\churn.csv")

In [4]:
for i in df.columns:
    print(i, len(df[i].unique()))
    print(df[i].unique())

customerID 7043
['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender 2
['Female' 'Male']
SeniorCitizen 2
[0 1]
Partner 2
['Yes' 'No']
Dependents 2
['No' 'Yes']
tenure 73
[ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39]
PhoneService 2
['No' 'Yes']
MultipleLines 3
['No phone service' 'No' 'Yes']
InternetService 3
['DSL' 'Fiber optic' 'No']
OnlineSecurity 3
['No' 'Yes' 'No internet service']
OnlineBackup 3
['Yes' 'No' 'No internet service']
DeviceProtection 3
['No' 'Yes' 'No internet service']
TechSupport 3
['No' 'Yes' 'No internet service']
StreamingTV 3
['No' 'Yes' 'No internet service']
StreamingMovies 3
['No' 'Yes' 'No internet service']
Contract 3
['Month-to-month' 'One year' 'Two year']
PaperlessBilling 2
['Yes' 'No']
PaymentMethod 4
['Electronic check' 'Mailed check' 'Ban

In [5]:
df["Churn"] = df["Churn"].replace({"No": 0, "Yes": 1})
y = df["Churn"]
x = df.drop(["Churn", "customerID"], axis=1)
columns = x.columns
x["TotalCharges"] = pd.to_numeric(x["TotalCharges"], errors='coerce')


C:\Users\dell\AppData\Local\Temp\ipykernel_13256\2339027411.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Churn"] = df["Churn"].replace({"No": 0, "Yes": 1})


In [6]:
cat_cols = x.select_dtypes(include=["category", "object"]).columns.tolist()
cat_cols.append("SeniorCitizen")

num_cols = [
    col for col in x.columns
    if col not in cat_cols
    and x[col].nunique() > 15
]
trimmer = OutlierTrimmer(
        capping_method="iqr",
        tail="both",
        fold=1.5
    )
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("outlier_capping", trimmer),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])
x = preprocessor.fit_transform(x)


In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [8]:
# yoooo logistic regression works better
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

models = {
    "LogisticRegression": LogisticRegression(),
    "RandomForest": RandomForestClassifier(),
    "DecisionTree": DecisionTreeClassifier(),
    "XGBoost":  XGBClassifier()

}

# Evaluate each model with cross-validation
for name, model in models.items():
    model.fit(x_train, y_train)
    preds = model.predict(x_test)
    report = classification_report(y_test, preds)
    print(name)
    print(report)

LogisticRegression
              precision    recall  f1-score   support

           0       0.86      0.89      0.87      1043
           1       0.65      0.57      0.61       366

    accuracy                           0.81      1409
   macro avg       0.76      0.73      0.74      1409
weighted avg       0.80      0.81      0.81      1409

RandomForest
              precision    recall  f1-score   support

           0       0.83      0.91      0.87      1043
           1       0.64      0.48      0.55       366

    accuracy                           0.80      1409
   macro avg       0.74      0.69      0.71      1409
weighted avg       0.78      0.80      0.78      1409

DecisionTree
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1043
           1       0.49      0.49      0.49       366

    accuracy                           0.74      1409
   macro avg       0.66      0.66      0.66      1409
weighted avg       0.74      

In [9]:
def objective(trial, X, y):
    penalty = trial.suggest_categorical(
        "penalty", ["l1", "l2", "elasticnet"]
    )

    param_grid = {
        "C": trial.suggest_float("C", 1e-3, 10, log=True),
        "max_iter": trial.suggest_int("max_iter", 100, 1000),
        "penalty": penalty,
        "class_weight": trial.suggest_categorical(
            "class_weight", [None, "balanced"]
        ),
    }

    if penalty == "elasticnet":
        param_grid["l1_ratio"] = trial.suggest_float("l1_ratio", 0.05, 0.9)
        param_grid["solver"] = "saga"
    elif penalty == "l1":
        param_grid["solver"] = "saga"
    else:
        param_grid["solver"] = "lbfgs"

    model = LogisticRegression(**param_grid)

    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="precision_weighted"
    )

    return scores.mean()
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: objective(trial, x, y), n_trials=100)

[I 2026-01-18 20:10:39,982] A new study created in memory with name: no-name-dfffc499-bb13-40a4-9002-78eb18685f13
[I 2026-01-18 20:10:40,446] Trial 0 finished with value: 0.8013468146190924 and parameters: {'penalty': 'l2', 'C': 0.01684019371029791, 'max_iter': 269, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.8013468146190924.
[I 2026-01-18 20:10:40,980] Trial 1 finished with value: 0.797774769847601 and parameters: {'penalty': 'elasticnet', 'C': 0.0029914703194343184, 'max_iter': 119, 'class_weight': 'balanced', 'l1_ratio': 0.2840275759680812}. Best is trial 0 with value: 0.8013468146190924.
[I 2026-01-18 20:10:41,583] Trial 2 finished with value: 0.7941211253475904 and parameters: {'penalty': 'l1', 'C': 0.3136626424485214, 'max_iter': 522, 'class_weight': None}. Best is trial 0 with value: 0.8013468146190924.
[I 2026-01-18 20:10:41,830] Trial 3 finished with value: 0.7931968863487389 and parameters: {'penalty': 'l2', 'C': 0.04618931915946542, 'max_iter': 771, 'class_we

In [10]:
print(study.best_value)
print(study.best_params)

0.8069342892170861
{'penalty': 'l2', 'C': 4.820422658540756, 'max_iter': 507, 'class_weight': 'balanced'}


In [11]:
"""current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

# Convert to proper MLflow URI
tracking_uri = f"file:///{os.path.join(parent_dir, "mlruns").replace(os.sep, '/')}"  # Use forward slashes

mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("churn_logistic_regression")

with mlflow.start_run():
    params = {'penalty': 'l2', 'C': 0.34379875424934764, 'max_iter': 540, 'class_weight': 'balanced'}
    l_r = LogisticRegression(**params)
    l_r.fit(x_train, y_train)
    preds = l_r.predict(x_test)

    # Metrics
    precision = precision_score(y_test, preds)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, preds)

    mlflow.log_params(params)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1", f1)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.sklearn.log_model(l_r, "model")
"""

'current_dir = os.getcwd()\nparent_dir = os.path.dirname(current_dir)\n\n# Convert to proper MLflow URI\ntracking_uri = f"file:///{os.path.join(parent_dir, "mlruns").replace(os.sep, \'/\')}"  # Use forward slashes\n\nmlflow.set_tracking_uri(tracking_uri)\nmlflow.set_experiment("churn_logistic_regression")\n\nwith mlflow.start_run():\n    params = {\'penalty\': \'l2\', \'C\': 0.34379875424934764, \'max_iter\': 540, \'class_weight\': \'balanced\'}\n    l_r = LogisticRegression(**params)\n    l_r.fit(x_train, y_train)\n    preds = l_r.predict(x_test)\n\n    # Metrics\n    precision = precision_score(y_test, preds)\n    recall = recall_score(y_test, preds)\n    f1 = f1_score(y_test, preds)\n    roc_auc = roc_auc_score(y_test, preds)\n\n    mlflow.log_params(params)\n    mlflow.log_metric("precision", precision)\n    mlflow.log_metric("recall", recall)\n    mlflow.log_metric("f1", f1)\n    mlflow.log_metric("roc_auc", roc_auc)\n    mlflow.sklearn.log_model(l_r, "model")\n'